# Bicycle Counts

Bicycle Counts (trip observations) merged with fixed-location Bicycle Counters to attach coordinates.  The merged Bicycle dataset provides point locations (lon/lat + WKT), site descriptors, timestamps, and count measures suitable for mapping trends, evaluating corridors, and supporting planning studies.
* Preprocessing: The geom_wkt column, which stored geometries as Well-Known Text (WKT) strings, contained some missing values (NaN) that would cause parsing errors. These were handled using on_invalid='ignore', which sets invalid or empty geometries to None rather than raising an exception. The geometry column was then parsed using GeoSeries.from_wkt(), a vectorized method significantly faster than row-wise alternatives like .apply(wkt.loads) for large datasets.
  

In [1]:
from datasets import load_dataset
import pandas as pd
import geopandas as gpd

# Load the dataset
dataset = load_dataset("oscur/NYC_bicycle_counts") 
# Select the training split
df = dataset["train"].to_pandas()

In [3]:
df.head()

,countid,id,date,counts,status,name,lon,lat,geom_wkt
0,19831896,100009425,06/03/2023 12:00:00 AM,27,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288)
1,19831897,100009425,06/03/2023 12:15:00 AM,9,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288)
2,19831898,100009425,06/03/2023 12:30:00 AM,14,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288)
3,19831899,100009425,06/03/2023 12:45:00 AM,10,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288)
4,19831900,100009425,06/03/2023 01:00:00 AM,5,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288)


In [5]:
print(f"Shape: {df.shape}")
print(df.dtypes)

Shape: (6911898, 9)
countid       int64
id            int64
date            str
counts        int64
status        int64
name            str
lon         float64
lat         float64
geom_wkt        str
dtype: object


## Preprocessing
Some values in geom_wkt are NaN (float). Fix by dropping or handling nulls before converting:

In [10]:
# Check how many nulls
print(df['geom_wkt'].isna().sum())

df_clean = df.dropna(subset=['geom_wkt'])


27675


# Generate geodataframe

In [12]:
from shapely import wkt

gdf = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.GeoSeries.from_wkt(df_clean['geom_wkt']),  # vectorized, fast
    crs='EPSG:4326'
)


In [14]:
gdf.head()

,countid,id,date,counts,status,name,lon,lat,geom_wkt,geometry
0,19831896,100009425,06/03/2023 12:00:00 AM,27,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288),POINT (-73.97138 40.67129)
1,19831897,100009425,06/03/2023 12:15:00 AM,9,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288),POINT (-73.97138 40.67129)
2,19831898,100009425,06/03/2023 12:30:00 AM,14,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288),POINT (-73.97138 40.67129)
3,19831899,100009425,06/03/2023 12:45:00 AM,10,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288),POINT (-73.97138 40.67129)
4,19831900,100009425,06/03/2023 01:00:00 AM,5,0,Prospect Park West,-73.971382,40.671288,POINT (-73.971382 40.671288),POINT (-73.97138 40.67129)


In [15]:
gdf.to_file("./geo_datasets/NYC_bicycle_counts.geojson", driver="GeoJSON")

# Visualize

In [16]:
import folium
from folium.plugins import MarkerCluster

# Sample before plotting - adjust n based on performance
gdf_sample = gdf.dropna(subset=['geometry']).sample(n=5000, random_state=42)


# Create base map centered around NYC
m = folium.Map(location=[40.71, -73.95], zoom_start=12)

# Add a marker cluster for better performance
marker_cluster = MarkerCluster().add_to(m)

# Iterate over GeoDataFrame rows
for idx, row in gdf_sample.iterrows():
    # Extract coordinates from geometry
    lat = row.geometry.y
    lon = row.geometry.x
    
    # Build tooltip content
    tooltip_text = (
        f"Name: {row['name']}<br>"
        f"Date: {row['date']}<br>"
        f"Count: {row['counts']}<br>"
        f"Status: {row['status']}"
    )
    
    # Add marker
    folium.Marker(
        location=[lat, lon],
        tooltip=tooltip_text
    ).add_to(marker_cluster)

# Display map
m